In [0]:
%run ./world_export_config

In [0]:
# Read Silver layer

silver_path = f"abfss://{CONTAINER}@{storage_account}.dfs.core.windows.net/silver/world_exports_cleaned"
gold_path = f"abfss://{CONTAINER}@{storage_account}.dfs.core.windows.net/gold/world_exports_processed"

silver_df = spark.read \
    .format("delta") \
    .load(silver_path)

print(f"Silver records loaded: {silver_df.count()}")

In [0]:
# Gold KPI 1: By Country:
from pyspark.sql import functions as F

silver_df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .format('delta') \
    .load(silver_path)


# Aggregate data for the Dashboard
gold_df = silver_df.groupBy("country", "year", "product_category") \
    .agg(
        F.sum("export_value_usd").alias("total_export_value"),
        F.avg("growth_rate_pct").alias("avg_growth_rate"),
        F.count("product_category").alias("transaction_count")
    )

# Save this to a 'processed' folder
gold_df.write \
    .mode("overwrite") \
    .option("header", "true") \
    .format('delta') \
    .save(gold_path)

gold_df.display(10)

In [0]:
gold_df.write \
    .mode("overwrite") \
    .format('delta') \
    .saveAsTable("world_exports_dashboard")